In [9]:
%load_ext autoreload
%autoreload 2
from PyPDF2 import PdfReader
import os
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from kg.pipeline_a.SentenceFormatterManager import SentenceFormatterManager
import sys, importlib,os
sys.modules.pop('PatentTextFormatter', None)
importlib.invalidate_caches()



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

# --- Text extraction utils ---
def extract_text_from_pdf(path: str) -> str:
    text = ""

    # Try PyPDF2 first
    try:
        from PyPDF2 import PdfReader
        reader = PdfReader(path)
        for page in reader.pages:
            page_text = page.extract_text() or ""
            text += page_text
    except ModuleNotFoundError:
        # Fall back to pdfplumber if available
        try:
            import pdfplumber
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text() or ""
                    text += page_text
        except ModuleNotFoundError as e:
            raise RuntimeError(
                "No PDF text extractor found. Install one of:\n"
                "  pip install PyPDF2\n"
                "  or\n"
                "  pip install pdfplumber"
            ) from e

    # Light cleanup
    return text.replace("\u00A0", " ").strip()


# --- Extract text from your PDF ---
pdf_path = "example.pdf"
large_text = extract_text_from_pdf(pdf_path)
if not large_text:
    raise RuntimeError(
        "No text was extracted. If the PDF is scanned images, use OCR (e.g., `pip install pytesseract pillow`)"
    )

# --- Initialize KGGen ---
from kg_gen import KGGen
api_key = os.getenv("GOOGLE_API_KEY")  # ensure this is set in your env

kg = KGGen(
    model="gemini/gemini-2.5-flash",
    temperature=0.0,
    api_key=api_key
)

# --- Generate graph ---
graph_1 = kg.generate(
    input_data=large_text,
    context="Family relationships",
    chunk_size=5000,   # characters the library will chunk; OK for long docs
    cluster=True
)

# --- Visualize ---
KGGen.visualize(graph_1, "", open_in_browser=True)

# Optional: quick sanity check
print("Extracted characters:", len(large_text))
print("Preview:", large_text[:400])


In [8]:
formatterManager = SentenceFormatterManager()
split = formatterManager.split("A computing apparatus configured to dynamically allocate processing resources among a plurality of concurrent machine-learning inference tasks based on predictive workload analysis, wherein said allocation is continuously optimized through feedback derived from real-time performance metrics and latency constraints.")
print("split" + str(split))
for i in split:
    simplify = formatterManager.simplify(i)
    print("simplify" + str(simplify))

ValueError: Unexpected JSON schema: ['A computing apparatus configured to dynamically allocate processing resources among a plurality of concurrent machine-learning inference tasks based on predictive workload analysis, wherein said allocation is continuously optimized through feedback derived from real-time performance metrics and latency constraints.']